In [ ]:
# Example 1: Instantiate the LoggingInstrumentor and verify its public API.
from opentelemetry.instrumentation.logging import LoggingInstrumentor

instrumentor = LoggingInstrumentor()
print(f"instrumentor = {instrumentor!r}")
print(f"type = {type(instrumentor).__name__}")

# Verify the instrumentor exposes the expected BaseInstrumentor interface.
for attr in ("instrument", "uninstrument", "instrumentation_dependencies"):
    assert hasattr(instrumentor, attr), f"missing attribute: {attr}"
    print(f"  has {attr}: {callable(getattr(instrumentor, attr))}")

deps = list(instrumentor.instrumentation_dependencies())
print(f"instrumentation_dependencies = {deps}")


In [ ]:
# Example 2: Inspect the default LogRecord format constants exposed by the package.
from opentelemetry.instrumentation.logging import constants

default_fmt = constants.DEFAULT_LOGGING_FORMAT
print(f"DEFAULT_LOGGING_FORMAT = {default_fmt!r}")

# The default format should reference the OpenTelemetry trace/span fields.
for token in ("otelTraceID", "otelSpanID", "otelServiceName"):
    assert token in default_fmt, f"expected {token} in default format"
    print(f"  contains {token}: True")

public_constants = [n for n in dir(constants) if not n.startswith('_')]
print(f"public constants: {public_constants}")


In [ ]:
# Example 3: Instrument with set_logging_format=True and inspect the resulting root format.
import logging
from opentelemetry.instrumentation.logging import LoggingInstrumentor

instrumentor = LoggingInstrumentor()
try:
    instrumentor.instrument(
        set_logging_format=True,
        logging_format="%(otelTraceID)s %(otelSpanID)s %(message)s",
        log_level=logging.WARNING,
    )
    root = logging.getLogger()
    assert root.level == logging.WARNING, f"unexpected root level: {root.level}"
    print(f"root level = {logging.getLevelName(root.level)}")

    # Format a record through the basicConfig'd handler.
    rec = root.makeRecord("t", logging.WARNING, "f", 1, "hi", (), None)
    formatted = root.handlers[0].format(rec)
    print(f"formatted = {formatted!r}")
    assert "hi" in formatted
finally:
    instrumentor.uninstrument()


In [ ]:
# Example 4: LoggingInstrumentor is a singleton; verify entry point + idempotent uninstrument.
from opentelemetry.instrumentation.logging import LoggingInstrumentor
from opentelemetry.instrumentation.instrumentor import BaseInstrumentor

a = LoggingInstrumentor()
b = LoggingInstrumentor()
assert a is b, "LoggingInstrumentor should be a singleton"
assert isinstance(a, BaseInstrumentor), "must subclass BaseInstrumentor"
print(f"singleton OK: {a is b}")

# Double-instrument / double-uninstrument should be tolerated.
a.instrument(set_logging_format=False)
a.instrument(set_logging_format=False)  # idempotent re-instrument
a.uninstrument()
a.uninstrument()  # idempotent re-uninstrument
print("idempotent instrument/uninstrument OK")

assert a.instrumentation_dependencies() is not None
print("all assertions passed")
